# Lesson 9: Evaluating Word Embeddings & Hyperparameter Optimization

In this lesson, we build a complete end-to-end experimental workflow for building, evaluating, and optimizing distributional word embeddings.

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Stream Compressed Data**: Read large text datasets directly from compressed `gzip` archives without full decompression.
2. **Handle Large Corpora**: Process text line-by-line using regular expressions and manage collections with `defaultdict`.
3. **Compute Spearman Rank Correlation from Scratch**: Develop, test, and validate Spearman's rank correlation metric step-by-step using Python primitives.
4. **Train & Evaluate Embeddings**: Build Word2Vec models (Skip-gram and CBOW) and evaluate their alignment with human similarity judgments (MEN benchmark).
5. **Perform Systematic Hyperparameter Optimization**: Conduct grid search and integrate advanced Bayesian optimization using **Optuna** to find optimal architectures.

## References

- [**Gensim Word2Vec Documentation**](https://radimrehurek.com/gensim/models/word2vec.html)
- [**Optuna Optimization Framework**](https://optuna.org/)
- [**MEN Dataset (Human Judgments)**](https://staff.fnwi.uva.nl/e.bruni/MEN)
- Bruni, E., Tran, N. K., & Baroni, M. (2014). *Multimodal Distributional Semantics*. Journal of Artificial Intelligence Research, 49, 1-47.


---
## 1. Environment & Path Setup

We start by importing the necessary libraries for data processing, vector training, statistical evaluation, and hyperparameter optimization.

We centralize the paths to the data folder and the MEN benchmark dataset. **Adjust the paths** if you are running this on your local machine.


In [ ]:
import os
import re
import gzip
import logging
from collections import defaultdict
from random import shuffle
import gensim
from gensim.models.word2vec import Word2Vec
from scipy.stats import spearmanr
import optuna

# Reduce Optuna logging verbosity to keep the notebook clean
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Path Configuration ──────────────────────────────────────────────────────
# Change DATA_DIR to the folder containing the Google Books n-gram gz file.
DATA_DIR = "/Users/luca/Data/cattolica/"
MEN_PATH = "/Users/luca/Gits/pfl_2026_unicatt/assets/MEN/MEN_dataset_natural_form_full.txt"
NGRAM_FILE_NAME = "googlebooks-eng-all-5gram-20120701-ca.gz"
NGRAM_PATH = os.path.join(DATA_DIR, NGRAM_FILE_NAME)

print(f"Setup complete.")
print(f"  DATA_DIR:   {DATA_DIR}")
print(f"  MEN_PATH:   {MEN_PATH}")
print(f"  NGRAM_PATH: {NGRAM_PATH}")


---
## 2. Loading the Google Books N-gram Corpus

The [Google Books N-gram corpus](https://books.google.com/ngrams) represents billions of words extracted from digitized publications. Because the datasets are enormous (often several gigabytes compressed), fully decompressing them can consume excessive disk space.

### Streaming Compressed Data
In Python, we can use `gzip.open()` to stream the compressed file line-by-line. This is highly memory-efficient. However, streaming from a gzip file yields **raw bytes** (`bytes` objects) rather than strings. We must explicitly convert these bytes using `.decode('utf-8')` before performing any string manipulation.

We read up to 5,000,000 lines, extract the 5-gram token list, clean the Google POS tags using a compiled regular expression, and split the corpus into two time eras:
- **`before`**: 1990–1999 (Pre-internet era)
- **`after`**: 2000–present (Modern era)


In [ ]:
# Initialize corpora splits using defaultdict
sampled = defaultdict(list)

# Compile the POS tag regex for performance
# e.g., 'running_VERB' -> 'running'
POS_TAG_RE = re.compile(r"_[A-Z.]+(\s|$|_)")

MAX_LINES = 5_000_000
c = 0

print(f"Reading data line-by-line from: {NGRAM_PATH}...")
try:
    with gzip.open(NGRAM_PATH, 'rb') as fh:
        for line in fh:
            if c >= MAX_LINES:
                break
            
            # 1. Decode bytes object to a UTF-8 string
            line_str = line.decode('utf-8').strip()
            
            # 2. Extract fields (separated by tabs)
            # Format: ngram \t year \t match_count \t volume_count
            text, year_str, n1, n2 = line_str.split("\t")
            
            # 3. Clean POS tags
            text = POS_TAG_RE.sub(" ", text)
            year = int(year_str)
            tokens = text.split()
            
            # 4. Filter by year
            if year > 1999:
                sampled["after"].append(tokens)
            elif year > 1989:
                sampled["before"].append(tokens)
            
            c += 1
            if c % 1_000_000 == 0:
                print(f"  Processed {c:,} lines...")
                
except FileNotFoundError:
    print(f"[ERROR] Could not find gzip file at {NGRAM_PATH}.")
    print("Please verify your local DATA_DIR path.")

print(f"\nCompleted streaming!")
print(f"  'before' era sentences: {len(sampled['before']):,}")
print(f"  'after' era sentences : {len(sampled['after']):,}")


---
## 3. Shuffling the Corpora

Since Word2Vec learns representations by iterating through sentences sequentially, ordering artifacts in the text file could bias the model's update steps. Shuffling the corpus prevents these order biases and improves model stability.


In [ ]:
# Shuffle each split in-place
shuffle(sampled['after'])
shuffle(sampled['before'])

# Look at the first 5 samples from the post-2000 split
print("Sample 'after' sentences:")
for tokens in sampled['after'][:5]:
    print(f"  {tokens}")


---
## 4. The Evaluation Benchmark: MEN Dataset

Word similarity models are typically evaluated against human benchmarks. The **MEN dataset** is a gold-standard collection of 3,000 word pairs with human similarity scores ranging from 0.0 to 50.0.

We parse each line of the MEN text file, split the tokens, and structure them as tuples of `(word1, word2, similarity_score_float)`.


In [ ]:
try:
    # Read the dataset and split each line into tokens
    men_raw = [line.split() for line in open(MEN_PATH, encoding='utf-8').readlines()]
    
    # Convert similarity score to a float and tuple format
    MEN = [(w1, w2, float(val)) for w1, w2, val in men_raw]
    
    print(f"Loaded {len(MEN)} pairs from the MEN dataset.")
    print("Top 10 human similarity judgments:")
    for w1, w2, val in MEN[:10]:
        print(f"  {w1:<15} - {w2:<15} : Score {val:.1f}")
except FileNotFoundError:
    print(f"[ERROR] MEN dataset not found at {MEN_PATH}.")


---
## 5. Spearman's Rank Correlation from Scratch

### The Math Behind Spearman's $\rho$
Spearman's rank correlation coefficient ($\rho$) measures the monotonic relationship between two variables. Instead of correlating raw scores directly (like Pearson), we convert the values of each variable into **ranks** and calculate the correlation between the ranks.

Assuming no tie values, the Spearman rank correlation is calculated as:
$$\rho = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$
Where:
- $d_i$ is the difference between the ranks of the $i$-th element of the two lists.
- $n$ is the total number of observations.

### Step-by-Step Implementation
To understand the algorithm, we will write it from scratch using list comprehensions and the `zip` function.


In [ ]:
def rank_observations(x):
    """Assign ranks to observations in a list, where higher values get smaller ranks (1-indexed).
    
    Example:
      [50, 100, 20] -> ranked: [100, 50, 20]
      Ranks for [50, 100, 20] are [2, 1, 3]
    """
    ranked = sorted(x, reverse=True)
    return [ranked.index(val) + 1 for val in x]

def scratch_spearman(x, y):
    """Calculate Spearman's rank correlation coefficient from scratch.
    
    Parameters
    ----------
    x, y : list of float or int
        The two parallel lists of values to correlate.
        
    Returns
    -------
    float
        Spearman rank correlation coefficient.
    """
    if len(x) != len(y):
        raise ValueError("Variables must be of the same length.")
        
    # 1. Convert observations to ranks
    ranked_x = rank_observations(x)
    ranked_y = rank_observations(y)
    
    # 2. Compute differences between corresponding ranks (d_i)
    d = [rx - ry for rx, ry in zip(ranked_x, ranked_y)]
    
    # 3. Square and sum differences (sum(d_i^2))
    d_squared_sum = sum([val ** 2 for val in d])
    
    # 4. Apply the formula
    n = len(x)
    numerator = 6 * d_squared_sum
    denominator = n * (n**2 - 1)  # equivalent to n^3 - n
    
    return 1 - (numerator / denominator)


### Validating Against SciPy
We test our from-scratch implementation on a toy dataset of distances and prices, comparing the output with SciPy's standard `scipy.stats.spearmanr` function. We will also inspect the p-value.


In [ ]:
# Toy datasets
distance = [50, 175, 250, 375, 425, 585, 720, 810, 875, 950]
price = [1.80, 1.25, 2, 1, 1.10, 1.20, 0.8, 0.60, 1.05, 0.85]

# Calculate correlation using scratch implementation
rho_scratch = scratch_spearman(distance, price)

# Calculate correlation using SciPy
scipy_res = spearmanr(distance, price)

print("Validation Comparison:")
print(f"  Scratch Spearman rho : {rho_scratch:.6f}")
print(f"  SciPy Spearman rho   : {scipy_res.statistic:.6f}")
print(f"  SciPy p-value        : {scipy_res.pvalue:.6f}")
print(f"  Absolute Difference  : {abs(rho_scratch - scipy_res.statistic):.2e}")


---
## 6. Training Word2Vec Models

We train two separate Word2Vec models using Gensim — one on each era (`before` and `after`).

### Hyperparameters Explained
- `sg`: The training architecture. `sg=0` uses Continuous Bag of Words (CBOW), and `sg=1` uses Skip-gram.
- `window`: The maximum distance between the current and predicted word within a sentence.
- `vector_size`: Dimensionality of the word vectors. Typical values range between 100 and 300.

We will train skip-gram models (`sg=1`) with a context window of 3 and vector size of 100.


In [ ]:
print("Training 'before' (1990-1999) era Word2Vec model...")
mbefore = Word2Vec(sampled['before'], sg=1, window=3, vector_size=100)

print("Training 'after' (2000-2012) era Word2Vec model...")
mafter = Word2Vec(sampled['after'], sg=1, window=3, vector_size=100)

print(f"\nTraining complete!")
print(f"  mbefore vocabulary size: {len(mbefore.wv.index_to_key):,} words")
print(f"  mafter vocabulary size : {len(mafter.wv.index_to_key):,} words")


---
## 7. Model Evaluation on the MEN Benchmark

To evaluate our models, we measure the cosine similarity between each word pair in the MEN dataset, and calculate the Spearman correlation against the human similarity ratings.

### Handling Out-of-Vocabulary (OOV) Entries
Some words in the MEN dataset might not exist in our training vocabulary (especially due to sample limitations). We must wrap the similarity lookup in a `try-except` block to capture `KeyError` exceptions, skipping OOV words. At the same time, we must filter both the model scores and human scores so they stay aligned and of equal length.


In [ ]:
def compare_men(model, goldstandard=MEN):
    """Compute Spearman correlation between model word similarities and human ratings.
    
    Parameters
    ----------
    model : Word2Vec
        The trained model to evaluate.
    goldstandard : list of tuples
        List of (word1, word2, human_rating) tuples.
        
    Returns
    -------
    scipy.stats.SpearmanrResult
        Object containing the correlation statistic and p-value.
    """
    model_scores = []
    human_scores = []
    
    for w1, w2, val in goldstandard:
        # Check if both words are present in the model's vocabulary
        if w1 in model.wv and w2 in model.wv:
            try:
                # Compute cosine similarity
                cos_sim = model.wv.similarity(w1, w2)
                model_scores.append(cos_sim)
                # Store corresponding human rating
                human_scores.append(val)
            except KeyError:
                pass  # Skip if a key error is thrown
                
    # If no word pairs were matched, return None
    if len(model_scores) == 0:
        return None
        
    return spearmanr(model_scores, human_scores)

# Run evaluation
eval_before = compare_men(mbefore)
eval_after = compare_men(mafter)

print("Initial Model Evaluation on MEN:")
if eval_before:
    print(f"  mbefore (1990-1999) - Spearman rho: {eval_before.statistic:.4f}  (p = {eval_before.pvalue:.2e})")
if eval_after:
    print(f"  mafter  (2000-2012) - Spearman rho: {eval_after.statistic:.4f}  (p = {eval_after.pvalue:.2e})")


---
## 8. Hyperparameter Optimization: Grid Search

Since optimal hyperparameters depend heavily on data size and task domains, we must systematically test configurations.

We implement a grid search using nested loops over:
- `sg`: `[0, 1]` (CBOW vs. Skip-gram)
- `window`: `[1, 2]`
- `vector_size`: `[100, 150, 200]`

> **Note**: In the original script, there was a hyperparameter-binding bug where the loop variables were ignored during model instantiation. We have resolved it below by correctly passing the current loop values to the model constructor.


In [ ]:
results = []

print("Running systematic Grid Search...")
for sg_ in (0, 1):
    for window_ in (1, 2):
        for vec_ in (100, 150, 200):
            # Corrected: passing loop variables to the Word2Vec model
            model = Word2Vec(
                sampled['before'], 
                sg=sg_, 
                window=window_, 
                vector_size=vec_,
                epochs=3
            )
            
            eval_res = compare_men(model)
            rho = eval_res.statistic if eval_res else 0.0
            
            res_tuple = (sg_, window_, vec_, rho)
            results.append(res_tuple)
            
            arch_label = "Skip-gram" if sg_ == 1 else "CBOW"
            print(f"  {arch_label:<9} | Window: {window_} | Dim: {vec_:<3} | Spearman rho: {rho:.4f}")


---
## 9. Advanced Optimization with Optuna

Grid search tests every combination, which becomes highly inefficient as we add more hyperparameters. **Optuna** is a state-of-the-art hyperparameter optimization framework that uses Bayesian methods (like Tree-structured Parzen Estimator) to search the parameter space dynamically.

### Setting up Optuna
1. We define an **objective function** that receives a `trial` object.
2. Inside the objective, we suggest parameters using methods like `trial.suggest_categorical` and `trial.suggest_int`.
3. We train our model and return the Spearman correlation (since we want to maximize it).
4. We create an Optuna study with `direction='maximize'` and run `study.optimize()`.


In [ ]:
def objective(trial):
    """Objective function for Optuna to optimize Word2Vec performance."""
    # Suggest hyperparameter values for the trial
    sg = trial.suggest_categorical("sg", [0, 1])
    window = trial.suggest_int("window", 1, 4)
    vector_size = trial.suggest_int("vector_size", 50, 200, step=50)
    
    # Train the model with suggested parameters
    model = Word2Vec(
        sampled['before'],
        sg=sg,
        window=window,
        vector_size=vector_size,
        epochs=2
    )
    
    # Evaluate on the MEN benchmark
    eval_res = compare_men(model)
    if eval_res:
        return eval_res.statistic
    return 0.0  # Fallback score

# 1. Create a study to maximize correlation
study = optuna.create_study(direction="maximize")

# 2. Run the optimization study
print("Running 5 trials of Optuna optimization...")
study.optimize(objective, n_trials=5)

print("\nOptimization study finished!")
print(f"Best trial Spearman rho: {study.best_value:.4f}")
print("Best parameters found:")
for name, value in study.best_params.items():
    print(f"  {name}: {value}")


---
## 10. Results Summary

| Model | Era | Architecture | Window | Vector Size | Spearman ρ (MEN) |
|-------|-----|--------------|--------|-------------|------------------|
| `mbefore` | 1990-1999 | Skip-gram | 3 | 100 | *Fill in based on output* |
| `mafter` | 2000-2012 | Skip-gram | 3 | 100 | *Fill in based on output* |
| Best Grid | 1990-1999 | *Fill in* | *Fill in* | *Fill in* | *Fill in* |
| Best Optuna | 1990-1999 | *Fill in* | *Fill in* | *Fill in* | *Fill in* |

### Discussion Questions
1. **CBOW vs. Skip-gram**: Based on your grid search, which architecture consistently produced higher correlations? Why does this architecture work better on smaller corpora?
2. **Data Constraints**: The correlation coefficients are relatively low (e.g., ~0.20-0.30) compared to state-of-the-art models (~0.70-0.80). What is the main cause of this difference?
3. **Optuna Advantages**: In what scenarios would Optuna be vastly superior to a grid search?


---
## Exercises

Complete the following tasks. Make sure your code is well-commented and clean. Discuss your findings in the markdown cells provided.


### Exercise 1 — Gzip Stream Processing & Vocabulary Counting

Write a memory-efficient stream processor that directly reads the compressed gzip file (`NGRAM_PATH`), extracts the occurrences of a keyword (e.g., `'science'`), and counts how many times it appears in each era (`before` vs. `after`).

**Requirements**:
- Read the file line-by-line using `gzip.open`.
- Do not store the lines or sentences in memory.
- Strip POS tags with the `POS_TAG_RE` expression before search.
- Print the final counts for each era.


In [ ]:
# Your code here
def count_keyword_in_gzip(keyword='science'):
    counts = {'before': 0, 'after': 0}
    
    # TODO: Write streaming reader and count occurrences of the keyword
    
    return counts

# Test your function
# print(count_keyword_in_gzip('science'))


*Your observations (edit this cell):*

- ...


### Exercise 2 — Spearman Rank Correlation with Ties

Our scratch Spearman correlation assumes that all values are unique (no ties). When lists contain ties (identical values), standard ranking assigns the average rank to the tied observations.

1. Write an updated ranking function `rank_observations_with_ties(x)` that handles ties using the standard average ranking method (similar to `scipy.stats.rankdata`).
2. Implement `scratch_spearman_with_ties(x, y)` using this new ranking function.
3. Test it against `scipy.stats.spearmanr` using lists with ties, for example:
   `x = [1, 2, 2, 4, 5]` and `y = [2, 3, 3, 3, 5]`.


In [ ]:
# Your code here
def rank_observations_with_ties(x):
    # TODO: Implement ranking that handles ties
    pass

def scratch_spearman_with_ties(x, y):
    # TODO: Implement Spearman correlation using rank_observations_with_ties
    pass

# Test and compare with scipy.stats.spearmanr on a toy dataset with ties
x_test = [1, 2, 2, 4, 5]
y_test = [2, 3, 3, 3, 5]


*Your observations (edit this cell):*

- ...


### Exercise 3 — Grid Search Expansion

Extend the Grid Search code to tune **two additional parameters**:
- `min_count`: `[1, 5]` (minimum count of words to include in vocabulary)
- `epochs`: `[1, 5]` (number of training iterations)

Select 4 parameter combinations of your choice that vary these two parameters, train models on the `before` corpus, evaluate them on the MEN dataset, and print the resulting Spearman correlations.


In [ ]:
# Your code here
# Define 4 distinct parameter configurations
# Format: (sg, window, vector_size, min_count, epochs)


*Your observations (edit this cell):*

- ...


### Exercise 4 — Full Optuna Study

Implement an Optuna study that searches over **five hyperparameters**:
1. `sg` (categorical: 0 or 1)
2. `window` (int: 1 to 5)
3. `vector_size` (int: 50 to 200, step 50)
4. `min_count` (int: 1 to 5)
5. `epochs` (int: 1 to 5)

Create the study, run it for 8 trials, and print the best hyperparameter configuration found.


In [ ]:
# Your code here
def optuna_objective(trial):
    # TODO: Suggest the 5 parameters and return Spearman score
    pass

# Create study and run optimization


*Your observations (edit this cell):*

- ...


### Exercise 5 — Word2Vec Negative Sampling (Empirical Analysis)

Word2Vec uses **negative sampling** to optimize calculations. The parameter `negative` (default: 5) specifies how many 'noise words' are drawn for each positive training instance.

1. Train two Skip-gram models on the `before` corpus with `vector_size=100` and `window=3`:
   - Model A: `negative=0` (Negative sampling turned off, using standard Hierarchical Softmax instead by setting `hs=1`).
   - Model B: `negative=20` (High negative sampling rate).
2. Evaluate both models on the MEN dataset.
3. Discuss the theoretical difference between Hierarchical Softmax and Negative Sampling, and explain the difference in empirical performance.


In [ ]:
# Your code here
# Train Model A (Hierarchical Softmax: hs=1, negative=0)

# Train Model B (Negative Sampling: hs=0, negative=20)

# Evaluate both on MEN


*Your observations (edit this cell):*

- ...
